# BreastDM 2D FCN Reproduction (PyTorch / Google Colab)

This notebook reproduces the **2D FCN-50 / FCN-101 breast-tumor segmentation baseline** reported by Zhao et al., *BreastDM: A DCE-MRI dataset for breast tumor image segmentation and classification* (2023).

### Paper-matched settings

- each tumor-containing 2D slice is an independent input
- spatial size: **224 × 224**
- batch size: **16**
- optimizer: **Adam**, initial learning rate **0.01**
- learning-rate factor: **0.1** after 10 epochs without improvement
- maximum training: **100 epochs**
- augmentation family: flipping, scaling, and clipping/cropping
- models: torchvision FCN with ResNet-50 or ResNet-101 backbone
- reported test results: FCN-50 **71.1% DSC / 77.7% mIoU / 77.0% PPV**; FCN-101 **71.4% / 79.0% / 76.2%**

### Reproduction decisions

The paper does not disclose the exact loss, pixel threshold, normalization, random seed, or checkpoint rule. This notebook makes those choices explicit: binary cross-entropy loss, a 0.5 threshold, ImageNet normalization/backbone initialization, seed 42, and best validation Dice checkpointing. MRI grayscale slices are repeated to three channels so ImageNet-pretrained FCN backbones can be used without changing their first layer.

This is research code, not a clinical diagnostic system.

In [ ]:
%pip install -q albumentations

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import csv, json, random, shutil, time
from pathlib import Path

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import ResNet50_Weights, ResNet101_Weights
from torchvision.models.segmentation import fcn_resnet50, fcn_resnet101
from tqdm.auto import tqdm

# Reproducible experiment configuration
SEED = 42
MODEL_NAME = "fcn_resnet50"  # Change to "fcn_resnet101" to reproduce FCN-101.
IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2
MAX_EPOCHS = 100
LEARNING_RATE = 1e-2
WEIGHT_DECAY = 0.0
THRESHOLD = 0.5
USE_PRETRAINED_BACKBONE = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

SOURCE_ZIP = Path("/content/drive/MyDrive/BreastDM_Project/data/segmentation_DS_7_14_2026.zip")
DATASET_ROOT = Path("/content/segmentation_DS_7_14_2026")
RUN_DIR = Path("/content/drive/MyDrive/BreastDM_Project/FCN_runs") / MODEL_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
BEST_PATH = RUN_DIR / "best_model.pt"
RECOVERY_PATH = RUN_DIR / "recovery_checkpoint.pt"
HISTORY_PATH = RUN_DIR / "training_history.csv"
RESULTS_PATH = RUN_DIR / "test_results.json"

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
assert DEVICE.type == "cuda", "In Colab, select Runtime > Change runtime type > GPU."
if not DATASET_ROOT.is_dir():
    assert SOURCE_ZIP.is_file(), f"Missing dataset ZIP: {SOURCE_ZIP}"
    shutil.unpack_archive(str(SOURCE_ZIP), "/content")
assert DATASET_ROOT.is_dir(), f"Dataset root not found after extraction: {DATASET_ROOT}"
print("Device:", DEVICE, "| model:", MODEL_NAME, "| output:", RUN_DIR)

## Dataset and leakage checks

Expected extracted layout:

```text
segmentation_DS_7_14_2026/
  train/images/  train/masks/
  val/images/    val/masks/
  test/images/   test/masks/
```

Filenames must follow `patient__sequence__slice.ext`. The assertions below ensure that no patient occurs in more than one split.

In [ ]:
class BreastDMSegmentationDataset(Dataset):
    EXTENSIONS = {".jpg", ".jpeg", ".png"}

    def __init__(self, root, split, transform=None):
        if split not in {"train", "val", "test"}:
            raise ValueError(f"Invalid split: {split}")
        self.split, self.transform = split, transform
        image_dir = Path(root) / split / "images"
        mask_dir = Path(root) / split / "masks"
        if not image_dir.is_dir() or not mask_dir.is_dir():
            raise FileNotFoundError(f"Missing image or mask directory for {split}")
        images = {p.stem: p for p in image_dir.iterdir() if p.suffix.lower() in self.EXTENSIONS}
        masks = {p.stem: p for p in mask_dir.iterdir() if p.suffix.lower() in self.EXTENSIONS}
        missing_masks, missing_images = set(images) - set(masks), set(masks) - set(images)
        if missing_masks or missing_images:
            raise ValueError(f"Pairing error: {len(missing_masks)} images lack masks; {len(missing_images)} masks lack images")
        self.samples = [(images[s], masks[s], s) for s in sorted(images)]
        if not self.samples:
            raise ValueError(f"No paired samples found in {split}")

    def __len__(self):
        return len(self.samples)

    @property
    def patient_ids(self):
        return {sample_id.split("__", 1)[0] for _, _, sample_id in self.samples}

    def __getitem__(self, index):
        image_path, mask_path, sample_id = self.samples[index]
        image = np.asarray(Image.open(image_path).convert("L"), dtype=np.uint8)
        mask = np.asarray(Image.open(mask_path).convert("L"), dtype=np.uint8)
        if image.shape != mask.shape:
            raise ValueError(f"Image/mask shape mismatch for {sample_id}: {image.shape} vs {mask.shape}")
        if self.transform:
            transformed = self.transform(image=image, mask=mask)
            image, mask = transformed["image"], transformed["mask"]
        # Repeat the grayscale MRI channel to match the pretrained RGB backbone.
        image = torch.from_numpy(np.asarray(image).copy()).float().unsqueeze(0).repeat(3, 1, 1) / 255.0
        image = (image - torch.tensor([0.485, 0.456, 0.406])[:, None, None]) / torch.tensor([0.229, 0.224, 0.225])[:, None, None]
        mask = (torch.from_numpy(np.asarray(mask).copy()).float() > 0).float()
        return image, mask, {"sample_id": sample_id, "patient_id": sample_id.split("__", 1)[0]}

# Scaling + crop returns exactly 224x224; flips match the augmentation family in the paper.
train_transform = A.Compose([
    A.SmallestMaxSize(max_size=240, p=1.0),
    A.RandomScale(scale_limit=0.10, p=0.5),
    A.PadIfNeeded(min_height=IMAGE_SIZE, min_width=IMAGE_SIZE, border_mode=cv2.BORDER_CONSTANT),
    A.RandomCrop(IMAGE_SIZE, IMAGE_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
])
eval_transform = A.Compose([A.Resize(IMAGE_SIZE, IMAGE_SIZE)])

train_dataset = BreastDMSegmentationDataset(DATASET_ROOT, "train", train_transform)
val_dataset = BreastDMSegmentationDataset(DATASET_ROOT, "val", eval_transform)
test_dataset = BreastDMSegmentationDataset(DATASET_ROOT, "test", eval_transform)

assert train_dataset.patient_ids.isdisjoint(val_dataset.patient_ids), "Train/val patient leakage"
assert train_dataset.patient_ids.isdisjoint(test_dataset.patient_ids), "Train/test patient leakage"
assert val_dataset.patient_ids.isdisjoint(test_dataset.patient_ids), "Val/test patient leakage"

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)

generator = torch.Generator().manual_seed(SEED)
common = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
    worker_init_fn=seed_worker,
)
train_loader = DataLoader(train_dataset, shuffle=True, generator=generator, **common)
val_loader = DataLoader(val_dataset, shuffle=False, **common)
test_loader = DataLoader(test_dataset, shuffle=False, **common)

print("Samples:", {k: len(v) for k, v in {"train": train_dataset, "val": val_dataset, "test": test_dataset}.items()})
print("Patients:", {k: len(v.patient_ids) for k, v in {"train": train_dataset, "val": val_dataset, "test": test_dataset}.items()})
x, y, metadata = next(iter(train_loader))
assert x.shape[1:] == (3, IMAGE_SIZE, IMAGE_SIZE)
assert y.shape[1:] == (IMAGE_SIZE, IMAGE_SIZE)
print("Batch:", x.shape, y.shape)

## FCN-50 / FCN-101

Torchvision's implementation follows the FCN semantic-segmentation design and uses a ResNet-50 or ResNet-101 backbone. The classifier is replaced with a binary tumor/background output head. Auxiliary output is disabled because the paper does not describe an auxiliary loss.

In [ ]:
def build_model(name, pretrained_backbone=True):
    if name == "fcn_resnet50":
        backbone_weights = ResNet50_Weights.IMAGENET1K_V1 if pretrained_backbone else None
        model = fcn_resnet50(weights=None, weights_backbone=backbone_weights, num_classes=1, aux_loss=False)
    elif name == "fcn_resnet101":
        backbone_weights = ResNet101_Weights.IMAGENET1K_V1 if pretrained_backbone else None
        model = fcn_resnet101(weights=None, weights_backbone=backbone_weights, num_classes=1, aux_loss=False)
    else:
        raise ValueError("MODEL_NAME must be fcn_resnet50 or fcn_resnet101")
    return model

model = build_model(MODEL_NAME, USE_PRETRAINED_BACKBONE).to(DEVICE)
with torch.no_grad():
    probe = model(x[:2].to(DEVICE))["out"]
assert probe.shape == (2, 1, IMAGE_SIZE, IMAGE_SIZE)
print("Output:", probe.shape, "| trainable parameters:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## Loss and paper-compatible metrics

- **DSC** and **PPV** are foreground/tumor metrics.
- **mIoU** is the mean of background IoU and tumor IoU, matching the paper's definition over all categories.
- Empty-target slices are retained and handled safely. Aggregate confusion counts are used so tiny or empty masks do not dominate the score.

In [ ]:
criterion = nn.BCEWithLogitsLoss()

class ConfusionMeter:
    def __init__(self):
        self.tp = self.fp = self.fn = self.tn = 0.0

    @torch.no_grad()
    def update(self, logits, targets, threshold=THRESHOLD):
        pred = torch.sigmoid(logits) >= threshold
        target = targets[:, None].bool()
        self.tp += (pred & target).sum().item()
        self.fp += (pred & ~target).sum().item()
        self.fn += (~pred & target).sum().item()
        self.tn += (~pred & ~target).sum().item()

    def compute(self):
        eps = 1e-7
        dice = 2 * self.tp / (2 * self.tp + self.fp + self.fn + eps)
        tumor_iou = self.tp / (self.tp + self.fp + self.fn + eps)
        background_iou = self.tn / (self.tn + self.fp + self.fn + eps)
        ppv = self.tp / (self.tp + self.fp + eps)
        sensitivity = self.tp / (self.tp + self.fn + eps)
        return {"dice": dice, "miou": (tumor_iou + background_iou) / 2, "tumor_iou": tumor_iou, "ppv": ppv, "sensitivity": sensitivity}

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.1, patience=10)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

def run_epoch(loader, training):
    model.train(training)
    meter, total_loss, count = ConfusionMeter(), 0.0, 0
    for images, masks, _ in tqdm(loader, leave=False):
        images, masks = images.to(DEVICE, non_blocking=True), masks.to(DEVICE, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            with torch.amp.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(images)["out"]
                loss = criterion(logits, masks[:, None])
            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        count += batch_size
        meter.update(logits.detach(), masks)
    return {"loss": total_loss / count, **meter.compute()}

print("Training components ready.")

## Train

Set `RESUME_TRAINING=True` after a Colab disconnect. The recovery checkpoint, best checkpoint, and CSV history are saved to Drive after every epoch.

In [ ]:
RESUME_TRAINING = False
start_epoch, best_dice, history = 0, -1.0, []

if RESUME_TRAINING:
    checkpoint = torch.load(RECOVERY_PATH, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    scheduler.load_state_dict(checkpoint["scheduler"])
    scaler.load_state_dict(checkpoint["scaler"])
    start_epoch = checkpoint["epoch"]
    best_dice = checkpoint["best_dice"]
    history = checkpoint["history"]
    if checkpoint.get("generator_state") is not None:
        generator.set_state(checkpoint["generator_state"])
    print("Resuming at epoch", start_epoch + 1)

def write_history(rows):
    with HISTORY_PATH.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

for epoch in range(start_epoch, MAX_EPOCHS):
    began = time.time()
    train_metrics = run_epoch(train_loader, training=True)
    val_metrics = run_epoch(val_loader, training=False)
    scheduler.step(val_metrics["loss"])
    row = {
        "epoch": epoch + 1,
        "lr": optimizer.param_groups[0]["lr"],
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
        "minutes": (time.time() - began) / 60,
    }
    history.append(row)
    improved = val_metrics["dice"] > best_dice
    if improved:
        best_dice = val_metrics["dice"]
    state = {
        "epoch": epoch + 1,
        "model_name": MODEL_NAME,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "best_dice": best_dice,
        "history": history,
        "generator_state": generator.get_state(),
        "config": {"seed": SEED, "image_size": IMAGE_SIZE, "batch_size": BATCH_SIZE, "lr": LEARNING_RATE},
    }
    torch.save(state, RECOVERY_PATH)
    if improved:
        torch.save(state, BEST_PATH)
    write_history(history)
    print(
        f"Epoch {epoch + 1:03d} | lr {row['lr']:.2e} | "
        f"train loss {train_metrics['loss']:.4f} DSC {train_metrics['dice']:.4f} | "
        f"val loss {val_metrics['loss']:.4f} DSC {val_metrics['dice']:.4f} "
        f"mIoU {val_metrics['miou']:.4f} PPV {val_metrics['ppv']:.4f} | "
        f"{row['minutes']:.1f} min"
    )

print("Training complete. Best validation DSC:", best_dice)

## Locked test evaluation

Run this cell only after training/model selection. It loads the best validation checkpoint and evaluates the untouched test split once.

In [ ]:
best = torch.load(BEST_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(best["model"])
test_metrics = run_epoch(test_loader, training=False)
results = {
    "model": MODEL_NAME,
    "best_epoch": best["epoch"],
    "threshold": THRESHOLD,
    **test_metrics,
    "paper_reference": {
        "fcn_resnet50": {"dice": 0.711, "miou": 0.777, "ppv": 0.770},
        "fcn_resnet101": {"dice": 0.714, "miou": 0.790, "ppv": 0.762},
    }[MODEL_NAME],
}
with RESULTS_PATH.open("w") as handle:
    json.dump(results, handle, indent=2)
print(json.dumps(results, indent=2))

## Learning curves and qualitative predictions

In [ ]:
epochs = [row["epoch"] for row in history]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for key, title, axis in [("loss", "Loss", axes[0]), ("dice", "DSC", axes[1]), ("miou", "mIoU", axes[2])]:
    axis.plot(epochs, [row[f"train_{key}"] for row in history], label="train")
    axis.plot(epochs, [row[f"val_{key}"] for row in history], label="validation")
    axis.set_title(title)
    axis.set_xlabel("Epoch")
    axis.grid(alpha=0.3)
    axis.legend()
axes[1].set_ylim(0, 1)
axes[2].set_ylim(0, 1)
plt.tight_layout()
plt.savefig(RUN_DIR / "training_curves.png", dpi=200, bbox_inches="tight")
plt.show()

model.eval()
images, masks, metadata = next(iter(test_loader))
with torch.no_grad():
    probabilities = torch.sigmoid(model(images.to(DEVICE))["out"]).cpu()
predictions = probabilities >= THRESHOLD

n = min(4, len(images))
fig, axes = plt.subplots(n, 4, figsize=(14, 3.5 * n), squeeze=False)
mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
std = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
for i in range(n):
    display_image = (images[i] * std + mean).clamp(0, 1)[0]
    axes[i, 0].imshow(display_image, cmap="gray")
    axes[i, 0].set_title(metadata["sample_id"][i], fontsize=7)
    axes[i, 1].imshow(masks[i], cmap="gray")
    axes[i, 1].set_title("Ground truth")
    axes[i, 2].imshow(probabilities[i, 0], cmap="viridis", vmin=0, vmax=1)
    axes[i, 2].set_title("Tumor probability")
    axes[i, 3].imshow(display_image, cmap="gray")
    axes[i, 3].imshow(predictions[i, 0], cmap="Reds", alpha=0.45)
    axes[i, 3].set_title("FCN prediction")
    for axis in axes[i]:
        axis.axis("off")
plt.tight_layout()
plt.savefig(RUN_DIR / "test_predictions.png", dpi=200, bbox_inches="tight")
plt.show()

## Reproduction checklist

- Keep the test cell untouched until model selection is complete.
- Run both FCN-50 and FCN-101 as separate experiments by changing `MODEL_NAME`.
- Report the random seed, split patient counts, best epoch, threshold, and whether pretrained backbones were used.
- Expect some difference from the paper because its loss, split construction, normalization, threshold, and seed were not published.
- Do not interpret segmentation output as medical advice.